In [1]:
import requests
import pandas as pd
from io import StringIO

url = "https://www.ibecs.or.jp/CASBEE/certified_buld/data/CASBEE_certified_buld.txt"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Referer": "https://www.ibecs.or.jp/CASBEE/certified_buld/CASBEE_certified_buld_list.htm",
}

r = requests.get(url, headers=headers, timeout=60)
r.raise_for_status()

# UTF-8 with BOM
text = r.content.decode("utf-8-sig")

# It is TAB separated
df = pd.read_csv(StringIO(text), sep="\t")

print(len(df))
print(df.columns)

1168
Index(['{'], dtype='object')


In [2]:
print(repr(text[:300]))

'{\n"aaData":[\n["1","（一財）住宅・建築SDGs推進センター","IBEC-C0001-TC(b)","2005年日本国際博覧会長久手日本館","0001.pdf","","2005-03-11","中部地方整備局 営繕部 建築課長 小野寺幸治","愛知県愛知郡長久手町","展示施設(仮設)","CASBEE-短期使用（展示施設）（2004年版）／実施設計","S","2008-02-29","1"],\n["2","（一財）住宅・建築SDGs推進センター","IBEC-C0002-TC(b)","2005年日本国際博覧会瀬戸日本館","0002.pdf","","2005-03'


In [3]:
import requests, json
import pandas as pd

url = "https://www.ibecs.or.jp/CASBEE/certified_buld/data/CASBEE_certified_buld.txt"
headers = {
    "User-Agent": "Mozilla/5.0",
    "Referer": "https://www.ibecs.or.jp/CASBEE/certified_buld/CASBEE_certified_buld_list.htm",
}

r = requests.get(url, headers=headers, timeout=60)
r.raise_for_status()

obj = json.loads(r.content.decode("utf-8-sig"))
rows = obj["aaData"]

print("rows:", len(rows))
print("example row length:", len(rows[0]))

rows: 1165
example row length: 14


In [4]:
import pandas as pd

cols = [
    "row_no",
    "certifier",
    "cert_no",
    "building_name",
    "pdf_file",
    "pdf_url_extra",
    "eval_date",
    "evaluator",
    "location",
    "use",
    "tool_version",
    "rank",
    "other_date",
    "flag",
]

df = pd.DataFrame(rows, columns=cols)

# Filter Tokyo
tokyo = df[df["location"].astype(str).str.contains("東京都", na=False)].copy()
print("Tokyo rows:", len(tokyo))

# Save
tokyo.to_csv("casbee_tokyo.csv", index=False, encoding="utf-8-sig")

Tokyo rows: 360


In [5]:
tokyo[["cert_no","building_name","location","rank","eval_date"]].head(20)

,cert_no,building_name,location,rank,eval_date
3,IBEC-C0004-EB,ゲートシティ大崎,東京都品川区,S,2005-09-07
4,IBEC-C0005-NC(c),竹中工務店東京本店,東京都江東区,S,2005-09-07
9,IBEC-C0010-NC(ｃ),銀座三井ビルディング,東京都中央区,S,2006-05-09
12,IBEC-C0013-RN(ｂ),松田平田設計本社ビル,東京都港区,S,2006-08-28
13,IBEC-C0014-NC(c),日本橋三井タワー,東京都中央区,S,2006-10-10
23,IBEC-C0024-NC(c),イオンモールむさし村山ミュー,東京都武蔵村山市,A,2008-03-06
29,IBEC-C0030-NC(b),レクセルプラザ篠崎,東京都江戸川区,B+,2008-08-01
33,IBEC-C0034-NC(b),レクセルガーデン北綾瀬,東京都足立区,B+,2008-08-28
34,第ERICAS080002NC号,レクセル日暮里新築工事,東京都荒川区,B+,2008-09-09
47,IBEC-C0041-RN,学校法人東洋学園 本郷キャンパス4号館,東京都文京区,C,2008-12-22


In [6]:
tokyo.shape

(360, 14)

In [7]:
tokyo.info()

<class 'pandas.core.frame.DataFrame'>
Index: 360 entries, 3 to 1160
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   row_no         360 non-null    object
 1   certifier      360 non-null    object
 2   cert_no        360 non-null    object
 3   building_name  360 non-null    object
 4   pdf_file       360 non-null    object
 5   pdf_url_extra  360 non-null    object
 6   eval_date      360 non-null    object
 7   evaluator      360 non-null    object
 8   location       360 non-null    object
 9   use            360 non-null    object
 10  tool_version   360 non-null    object
 11  rank           360 non-null    object
 12  other_date     360 non-null    object
 13  flag           360 non-null    object
dtypes: object(14)
memory usage: 42.2+ KB


In [8]:
from deep_translator import GoogleTranslator
import pandas as pd

translator = GoogleTranslator(source='ja', target='en')

def translate_series(series):
    unique_vals = series.dropna().unique()
    mapping = {}
    for val in unique_vals:
        try:
            mapping[val] = translator.translate(val)
        except:
            mapping[val] = val  # fallback if API hiccups
    return series.map(mapping)

cols_to_translate = ["building_name", "location", "use"]

for col in cols_to_translate:
    tokyo[col + "_en"] = translate_series(tokyo[col])

tokyo.head()

,row_no,certifier,cert_no,building_name,pdf_file,pdf_url_extra,eval_date,evaluator,location,use,tool_version,rank,other_date,flag,building_name_en,location_en,use_en
3,4,（一財）住宅・建築SDGs推進センター,IBEC-C0004-EB,ゲートシティ大崎,0004.pdf,,2005-09-07,ゲートシティ大崎業務商業棟全体管理組合 理事長 丹羽守裕,東京都品川区,事務所・物販店・飲食店,CASBEE-既存（2004年版）／運用,S,2010-09-06,1,Gate City Osaki,"Shinagawa Ward, Tokyo","Offices, retail stores, restaurants"
4,5,（一財）住宅・建築SDGs推進センター,IBEC-C0005-NC(c),竹中工務店東京本店,0005.pdf,,2005-09-07,(株)竹中工務店 東京本店取締役本店長 羽田碩幸,東京都江東区,事務所,CASBEE-新築（2004年版）／竣工,S,2007-09-29,1,Takenaka Corporation Tokyo Main Branch,"Koto Ward, Tokyo",office
9,10,（一財）住宅・建築SDGs推進センター,IBEC-C0010-NC(ｃ),銀座三井ビルディング,0010.pdf,,2006-05-09,三井不動産(株) 代表取締役社長 岩沙 弘道,東京都中央区,事務所・ホテル,CASBEE-新築（2004年版）／竣工,S,2008-09-29,1,Ginza Mitsui Building,"Chuo Ward, Tokyo",Office/Hotel
12,13,（一財）住宅・建築SDGs推進センター,IBEC-C0013-RN(ｂ),松田平田設計本社ビル,0013.pdf,,2006-08-28,(株)松田平田設計 代表取締役社長 中園 正樹,東京都港区,事務所,CASBEE-改修（2005年版）／実施設計,S,2009-08-29,1,Matsuda Hirata Design Head Office Building,"Minato Ward, Tokyo",office
13,14,（一財）住宅・建築SDGs推進センター,IBEC-C0014-NC(c),日本橋三井タワー,0014.pdf,,2006-10-10,三井不動産(株) 代表取締役社長 岩沙 弘道,東京都中央区,事務所・ホテル,CASBEE-新築（2004年版）／竣工,S,2008-07-29,1,Nihonbashi Mitsui Tower,"Chuo Ward, Tokyo",Office/Hotel


In [9]:
tokyo.head()

,row_no,certifier,cert_no,building_name,pdf_file,pdf_url_extra,eval_date,evaluator,location,use,tool_version,rank,other_date,flag,building_name_en,location_en,use_en
3,4,（一財）住宅・建築SDGs推進センター,IBEC-C0004-EB,ゲートシティ大崎,0004.pdf,,2005-09-07,ゲートシティ大崎業務商業棟全体管理組合 理事長 丹羽守裕,東京都品川区,事務所・物販店・飲食店,CASBEE-既存（2004年版）／運用,S,2010-09-06,1,Gate City Osaki,"Shinagawa Ward, Tokyo","Offices, retail stores, restaurants"
4,5,（一財）住宅・建築SDGs推進センター,IBEC-C0005-NC(c),竹中工務店東京本店,0005.pdf,,2005-09-07,(株)竹中工務店 東京本店取締役本店長 羽田碩幸,東京都江東区,事務所,CASBEE-新築（2004年版）／竣工,S,2007-09-29,1,Takenaka Corporation Tokyo Main Branch,"Koto Ward, Tokyo",office
9,10,（一財）住宅・建築SDGs推進センター,IBEC-C0010-NC(ｃ),銀座三井ビルディング,0010.pdf,,2006-05-09,三井不動産(株) 代表取締役社長 岩沙 弘道,東京都中央区,事務所・ホテル,CASBEE-新築（2004年版）／竣工,S,2008-09-29,1,Ginza Mitsui Building,"Chuo Ward, Tokyo",Office/Hotel
12,13,（一財）住宅・建築SDGs推進センター,IBEC-C0013-RN(ｂ),松田平田設計本社ビル,0013.pdf,,2006-08-28,(株)松田平田設計 代表取締役社長 中園 正樹,東京都港区,事務所,CASBEE-改修（2005年版）／実施設計,S,2009-08-29,1,Matsuda Hirata Design Head Office Building,"Minato Ward, Tokyo",office
13,14,（一財）住宅・建築SDGs推進センター,IBEC-C0014-NC(c),日本橋三井タワー,0014.pdf,,2006-10-10,三井不動産(株) 代表取締役社長 岩沙 弘道,東京都中央区,事務所・ホテル,CASBEE-新築（2004年版）／竣工,S,2008-07-29,1,Nihonbashi Mitsui Tower,"Chuo Ward, Tokyo",Office/Hotel


In [10]:
tokyo = tokyo.drop(columns=["pdf_file","pdf_url_extra"])
tokyo.head()

,row_no,certifier,cert_no,building_name,eval_date,evaluator,location,use,tool_version,rank,other_date,flag,building_name_en,location_en,use_en
3,4,（一財）住宅・建築SDGs推進センター,IBEC-C0004-EB,ゲートシティ大崎,2005-09-07,ゲートシティ大崎業務商業棟全体管理組合 理事長 丹羽守裕,東京都品川区,事務所・物販店・飲食店,CASBEE-既存（2004年版）／運用,S,2010-09-06,1,Gate City Osaki,"Shinagawa Ward, Tokyo","Offices, retail stores, restaurants"
4,5,（一財）住宅・建築SDGs推進センター,IBEC-C0005-NC(c),竹中工務店東京本店,2005-09-07,(株)竹中工務店 東京本店取締役本店長 羽田碩幸,東京都江東区,事務所,CASBEE-新築（2004年版）／竣工,S,2007-09-29,1,Takenaka Corporation Tokyo Main Branch,"Koto Ward, Tokyo",office
9,10,（一財）住宅・建築SDGs推進センター,IBEC-C0010-NC(ｃ),銀座三井ビルディング,2006-05-09,三井不動産(株) 代表取締役社長 岩沙 弘道,東京都中央区,事務所・ホテル,CASBEE-新築（2004年版）／竣工,S,2008-09-29,1,Ginza Mitsui Building,"Chuo Ward, Tokyo",Office/Hotel
12,13,（一財）住宅・建築SDGs推進センター,IBEC-C0013-RN(ｂ),松田平田設計本社ビル,2006-08-28,(株)松田平田設計 代表取締役社長 中園 正樹,東京都港区,事務所,CASBEE-改修（2005年版）／実施設計,S,2009-08-29,1,Matsuda Hirata Design Head Office Building,"Minato Ward, Tokyo",office
13,14,（一財）住宅・建築SDGs推進センター,IBEC-C0014-NC(c),日本橋三井タワー,2006-10-10,三井不動産(株) 代表取締役社長 岩沙 弘道,東京都中央区,事務所・ホテル,CASBEE-新築（2004年版）／竣工,S,2008-07-29,1,Nihonbashi Mitsui Tower,"Chuo Ward, Tokyo",Office/Hotel


In [11]:
ward_map = {
    "Chiyoda": "Chiyoda",
    "Chuo": "Chuo",
    "Minato": "Minato",
    "Shinjuku": "Shinjuku",
    "Bunkyo": "Bunkyo",
    "Taito": "Taito",
    "Sumida": "Sumida",
    "Koto": "Koto",
    "Shinagawa": "Shinagawa",
    "Meguro": "Meguro",
    "Ota": "Ota",
    "Setagaya": "Setagaya",
    "Shibuya": "Shibuya",
    "Nakano": "Nakano",
    "Suginami": "Suginami",
    "Toshima": "Toshima",
    "Kita": "Kita",
    "Arakawa": "Arakawa",
    "Itabashi": "Itabashi",
    "Nerima": "Nerima",
    "Adachi": "Adachi",
    "Katsushika": "Katsushika",
    "Edogawa": "Edogawa",
}

def ward_from_en(loc):
    for w in ward_map:
        if w.lower() in loc.lower():
            return w
    return None

tokyo["tokyo_ward"] = tokyo["location_en"].apply(ward_from_en)

In [12]:
tokyo.head()

,row_no,certifier,cert_no,building_name,eval_date,evaluator,location,use,tool_version,rank,other_date,flag,building_name_en,location_en,use_en,tokyo_ward
3,4,（一財）住宅・建築SDGs推進センター,IBEC-C0004-EB,ゲートシティ大崎,2005-09-07,ゲートシティ大崎業務商業棟全体管理組合 理事長 丹羽守裕,東京都品川区,事務所・物販店・飲食店,CASBEE-既存（2004年版）／運用,S,2010-09-06,1,Gate City Osaki,"Shinagawa Ward, Tokyo","Offices, retail stores, restaurants",Shinagawa
4,5,（一財）住宅・建築SDGs推進センター,IBEC-C0005-NC(c),竹中工務店東京本店,2005-09-07,(株)竹中工務店 東京本店取締役本店長 羽田碩幸,東京都江東区,事務所,CASBEE-新築（2004年版）／竣工,S,2007-09-29,1,Takenaka Corporation Tokyo Main Branch,"Koto Ward, Tokyo",office,Koto
9,10,（一財）住宅・建築SDGs推進センター,IBEC-C0010-NC(ｃ),銀座三井ビルディング,2006-05-09,三井不動産(株) 代表取締役社長 岩沙 弘道,東京都中央区,事務所・ホテル,CASBEE-新築（2004年版）／竣工,S,2008-09-29,1,Ginza Mitsui Building,"Chuo Ward, Tokyo",Office/Hotel,Chuo
12,13,（一財）住宅・建築SDGs推進センター,IBEC-C0013-RN(ｂ),松田平田設計本社ビル,2006-08-28,(株)松田平田設計 代表取締役社長 中園 正樹,東京都港区,事務所,CASBEE-改修（2005年版）／実施設計,S,2009-08-29,1,Matsuda Hirata Design Head Office Building,"Minato Ward, Tokyo",office,Minato
13,14,（一財）住宅・建築SDGs推進センター,IBEC-C0014-NC(c),日本橋三井タワー,2006-10-10,三井不動産(株) 代表取締役社長 岩沙 弘道,東京都中央区,事務所・ホテル,CASBEE-新築（2004年版）／竣工,S,2008-07-29,1,Nihonbashi Mitsui Tower,"Chuo Ward, Tokyo",Office/Hotel,Chuo


In [18]:
from geopy.geocoders import Nominatim
import pandas as pd
import time

geolocator = Nominatim(user_agent="tokyo_green_project")

def geocode_address(address):
    try:
        location = geolocator.geocode(address)
        if location:
            return location.latitude, location.longitude
    except:
        return None, None

tokyo["full_address"] = tokyo["location_en"] + ", Tokyo, Japan"

tokyo[["latitude","longitude"]] = tokyo["full_address"].apply(
    lambda x: pd.Series(geocode_address(x))
)

time.sleep(1)  # avoid rate limiting

In [19]:
tokyo.head(20)

,row_no,certifier,cert_no,building_name,eval_date,evaluator,location,use,tool_version,rank,other_date,flag,building_name_en,location_en,use_en,tokyo_ward,full_address,latitude,longitude
3,4,（一財）住宅・建築SDGs推進センター,IBEC-C0004-EB,ゲートシティ大崎,2005-09-07,ゲートシティ大崎業務商業棟全体管理組合 理事長 丹羽守裕,東京都品川区,事務所・物販店・飲食店,CASBEE-既存（2004年版）／運用,S,2010-09-06,1,Gate City Osaki,"Shinagawa Ward, Tokyo","Offices, retail stores, restaurants",Shinagawa,"Shinagawa Ward, Tokyo, Tokyo, Japan",35.609201,139.730198
4,5,（一財）住宅・建築SDGs推進センター,IBEC-C0005-NC(c),竹中工務店東京本店,2005-09-07,(株)竹中工務店 東京本店取締役本店長 羽田碩幸,東京都江東区,事務所,CASBEE-新築（2004年版）／竣工,S,2007-09-29,1,Takenaka Corporation Tokyo Main Branch,"Koto Ward, Tokyo",office,Koto,"Koto Ward, Tokyo, Tokyo, Japan",35.654191,139.795259
9,10,（一財）住宅・建築SDGs推進センター,IBEC-C0010-NC(ｃ),銀座三井ビルディング,2006-05-09,三井不動産(株) 代表取締役社長 岩沙 弘道,東京都中央区,事務所・ホテル,CASBEE-新築（2004年版）／竣工,S,2008-09-29,1,Ginza Mitsui Building,"Chuo Ward, Tokyo",Office/Hotel,Chuo,"Chuo Ward, Tokyo, Tokyo, Japan",35.687838,139.789365
12,13,（一財）住宅・建築SDGs推進センター,IBEC-C0013-RN(ｂ),松田平田設計本社ビル,2006-08-28,(株)松田平田設計 代表取締役社長 中園 正樹,東京都港区,事務所,CASBEE-改修（2005年版）／実施設計,S,2009-08-29,1,Matsuda Hirata Design Head Office Building,"Minato Ward, Tokyo",office,Minato,"Minato Ward, Tokyo, Tokyo, Japan",35.633142,139.735781
13,14,（一財）住宅・建築SDGs推進センター,IBEC-C0014-NC(c),日本橋三井タワー,2006-10-10,三井不動産(株) 代表取締役社長 岩沙 弘道,東京都中央区,事務所・ホテル,CASBEE-新築（2004年版）／竣工,S,2008-07-29,1,Nihonbashi Mitsui Tower,"Chuo Ward, Tokyo",Office/Hotel,Chuo,"Chuo Ward, Tokyo, Tokyo, Japan",35.687838,139.789365
23,24,（一財）住宅・建築SDGs推進センター,IBEC-C0024-NC(c),イオンモールむさし村山ミュー,2008-03-06,イオンモール(株) 代表取締役社長 村上 教行,東京都武蔵村山市,物販店・飲食店・集会所,CASBEE-新築（2006年版）／竣工,A,2009-11-12,1,Aeon Mall Musashi Murayama Mu,"Musashimurayama City, Tokyo",Retail stores/restaurants/gathering halls,None,"Musashimurayama City, Tokyo, Tokyo, Japan",35.755188,139.387517
29,30,（一財）住宅・建築SDGs推進センター,IBEC-C0030-NC(b),レクセルプラザ篠崎,2008-08-01,扶桑レクセル（株） 代表取締役社長 中村 護,東京都江戸川区,集合住宅,CASBEE-新築（2006年版）／実施,B+,2011-03-05,1,Lexel Plaza Shinozaki,"Edogawa Ward, Tokyo",apartment complex,Edogawa,"Edogawa Ward, Tokyo, Tokyo, Japan",35.706632,139.868677
33,34,（一財）住宅・建築SDGs推進センター,IBEC-C0034-NC(b),レクセルガーデン北綾瀬,2008-08-28,扶桑レクセル（株） 代表取締役社長 中村 護,東京都足立区,集合住宅,CASBEE-新築（2006年版）／実施,B+,2011-01-23,1,Rexel Garden Kitaayase,"Adachi Ward, Tokyo",apartment complex,Adachi,"Adachi Ward, Tokyo, Tokyo, Japan",35.774603,139.804516
34,35,日本ERI(株),第ERICAS080002NC号,レクセル日暮里新築工事,2008-09-09,扶桑レクセル株式会社 代表取締役社長 中村護,東京都荒川区,集合住宅,CASBEE-新築（2006年版）／実施設計,B+,2011-08-28,1,Lexel Nippori new construction,"Arakawa Ward, Tokyo",apartment complex,Arakawa,"Arakawa Ward, Tokyo, Tokyo, Japan",35.737529,139.781310
47,48,（一財）住宅・建築SDGs推進センター,IBEC-C0041-RN,学校法人東洋学園 本郷キャンパス4号館,2008-12-22,学校法人東洋学園 理事長 江澤 雄一,東京都文京区,学校,CASBEE-改修（2006年版）／竣工,C,2013-12-21,1,Toyo Gakuen Educational Corporation Hongo Camp...,"Bunkyo Ward, Tokyo",school,Bunkyo,"Bunkyo Ward, Tokyo, Tokyo, Japan",35.538701,139.433440


In [20]:
tokyo["rank"].value_counts()

rank
A     164
S     139
B+     46
B-      6
C       2
Name: count, dtype: int64

In [21]:
import unicodedata

tokyo["rank"] = (
    tokyo["rank"]
    .astype("string")
    .fillna(pd.NA)
    .apply(lambda x: unicodedata.normalize("NFKC", x) if pd.notna(x) else x)
    .str.replace("\u3000", " ", regex=False)
    .str.strip()
    .str.upper()
)

hide_labels = {
    "(申請者の希望により非公開)",
    "申請者の希望により非公開",
    "非公開",
    "N/A",
    "NA",
    "NONE",
    ""
}
tokyo["rank"] = tokyo["rank"].replace(list(hide_labels), pd.NA)

tokyo["rank"].value_counts(dropna=False)

rank
A       164
S       139
B+       46
B-        6
<NA>      3
C         2
Name: count, dtype: int64

In [22]:
tokyo.to_csv("/Users/camillascandola/Desktop/tokyo-urban-relief-analysis/datasets/casbee_tokyo.csv", index=False, encoding="utf-8-sig")